# Performance evaluation — Comparable baseline

Run cells in order, or restart the kernel and **Run All**. Reusable helpers live in
`experiments/homework1/performance`; every execution creates a unique directory in
`outputs/homework1/performance/runs`. The notebook launches its own native Engine
and never attaches to or stops an existing server.

Activate the existing Python environment and source CANN **before** starting
Jupyter. Set `ASCEND_RT_VISIBLE_DEVICES` to one free physical chip. The notebook
itself orchestrates CPU-side work; the disposable Engine process runs the real
model and managed caches on that NPU.

Defaults: exact saved input IDs of lengths **128 and 2048**, target outputs
**32 and 128**, concurrency **1**, **3 warmups** of every shape in each timing
pass, then **10 measured repetitions** per case/pass. Generation ignores EOS
through the supported Engine API to measure fixed lengths. This is controlled
microbenchmarking, not natural stop-on-EOS serving.

The selected correctness run’s status is copied into this run. Any outstanding
failures remain visible; performance does not establish correctness.

## 1. Explicit configuration and saved inputs

The inputs are loaded from a saved correctness run without retokenizing, truncating,
or adding chat tokens. Each selected raw fixture already has the exact measured
length. Set `HOMEWORK_PERF_FIXTURES` to choose a specific `fixtures.json`; otherwise
the newest compatible saved run is selected and recorded. Checkpoint hashes must
match. Change the configuration here and restart/run all for a different matrix.

For a short infrastructure smoke run, explicitly set `HOMEWORK_PERF_OUTPUT_LENGTHS=2,3`,
`HOMEWORK_PERF_WARMUPS=1`, and `HOMEWORK_PERF_REPETITIONS=1`. Such a run is labelled
as reduced coverage and does not satisfy the ten-repetition baseline.

In [ ]:
import dataclasses
import json
import os
import sys
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "python/tokenspeed").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.homework1.performance.config import PerformanceConfig, workload_matrix
from experiments.homework1.performance.runner import (
    find_fixture_source,
    prepare_run,
    run_benchmark,
)
from experiments.homework1.evaluation.fixtures import load_fixtures

CHECKPOINT = Path(
    os.environ.get("HOMEWORK_PERF_MODEL", ROOT / "outputs/models/Qwen3.5-0.8B")
).resolve()
CONFIG = PerformanceConfig(
    input_lengths=tuple(
        int(n)
        for n in os.environ.get("HOMEWORK_PERF_INPUT_LENGTHS", "128,2048").split(",")
    ),
    output_lengths=tuple(
        int(n)
        for n in os.environ.get("HOMEWORK_PERF_OUTPUT_LENGTHS", "32,128").split(",")
    ),
    warmups=int(os.environ.get("HOMEWORK_PERF_WARMUPS", "3")),
    repetitions=int(os.environ.get("HOMEWORK_PERF_REPETITIONS", "10")),
    eos_policy=os.environ.get("HOMEWORK_PERF_EOS_POLICY", "fixed_length"),
    execution_mode="eager",
    context_length=4096,
    capacity=32768,
    seed=17,
    profile=os.environ.get("HOMEWORK_PERF_PROFILE", "1") == "1",
)
FIXTURE_SOURCE = (
    Path(os.environ["HOMEWORK_PERF_FIXTURES"]).resolve()
    if "HOMEWORK_PERF_FIXTURES" in os.environ
    else find_fixture_source(ROOT, input_lengths=CONFIG.input_lengths)
)
CORRECTNESS_SUMMARY = FIXTURE_SOURCE.parent / "summary.json"
RUN = prepare_run(
    ROOT,
    CHECKPOINT,
    FIXTURE_SOURCE,
    CONFIG,
    correctness_summary=CORRECTNESS_SUMMARY if CORRECTNESS_SUMMARY.exists() else None,
)
print("Run:", RUN)
print("Saved fixtures:", FIXTURE_SOURCE)
print(json.dumps(dataclasses.asdict(CONFIG), indent=2))
print(
    [
        (w.name, len(w.input_ids), w.output_tokens)
        for w in workload_matrix(CONFIG, load_fixtures(RUN / "fixtures.json"))
    ]
)
print("Linked correctness:", (RUN / "correctness_status.json").read_text())

## 2. Run isolated latency, forward, memory and profile passes

**Primary latency:** in-process streaming Engine client, including scheduler IPC
and detokenization. TTFT ends at the first nonempty generated-text event. No HTTP
transport is involved. TPOT uses first/last token-count arrival timestamps and
`N-1`; a coalesced first event makes TPOT unavailable. Raw cumulative token counts
and finish reasons are retained. No per-step synchronization is added here.

**Forward diagnostic:** a separate warmed pass synchronizes around the complete
`ModelExecutor._forward_step`. It includes model/logits/sampling and state work
inside that scope, plus Python/launch overhead; it excludes scheduler queueing,
input preparation, earlier attention metadata and response transport. It is
reported as forward wall time, not pure device-kernel time. Individual decode
step samples are saved.

**Memory/profile:** allocator counters are read in the NPU process. Device-wide
`npu-smi` snapshots and short/long CPU+NPU profiles run in separate requests after
warmup. Profiling never contributes primary timing samples. Every Engine call
creates a fresh request with prefix caching disabled. Control files are read at
prompt boundaries. Each request is followed by an unmeasured one-token control
request to flush buffered instrumentation/profiles before the next client timer.
These lightweight hooks still add some host overhead.

The worker log is written continuously to `RUN / "worker.log"`. If it fails or
times out, available raw samples and errors still reach the report.

In [ ]:
RESULT = run_benchmark(ROOT, RUN, timeout_seconds=7200, failure_policy="record")
print(json.dumps(RESULT, indent=2))

## 3. Results and variability

The table combines independent passes: request latency comes from primary
streaming requests; forward times come from synchronized diagnostic requests.
All values include explicit medians and min/max. Invalid and warmup requests
never enter those statistics. Counts below 10 remain visibly reduced coverage.
Review the actual output counts before comparing target lengths.

In [ ]:
from IPython.display import Markdown, display

display(Markdown((RUN / "report.md").read_text()))
print("Raw client/forward samples:", RUN / "samples.csv")
print("Machine-readable medians/min/max:", RUN / "statistics.csv")
print("Full per-request evidence:", RUN / "samples.json")

## 4. HBM definitions

Framework allocated/reserved counters describe this NPU process; device-wide
`npu-smi` includes other processes. Startup peaks precede the first request peak
reset. Warmed request peaks include model weights, preallocated KV/recurrent
pools and workspace, which are not separately attributed. A large startup pool
can keep HBM similar across different lengths. Units are bytes in raw snapshots
and MiB (`bytes / 2**20`) in summary tables.

Workload device-wide snapshots run in separate memory requests after the first
stream event. Their collection intervals are saved; if a short request ends
before the snapshot finishes, do not claim a precisely timed active HBM reading.
Unavailable tooling/APIs stay explicit rather than becoming zero memory.

In [ ]:
startup = RUN / "startup_memory.json"
print(
    startup.read_text()
    if startup.exists()
    else "NPU allocator snapshot unavailable: inspect worker.log"
)
for path in sorted(RUN.glob("npu_smi_*.json")):
    snapshot = json.loads(path.read_text())
    print(
        path.name, {k: v for k, v in snapshot.items() if k not in ("stdout", "stderr")}
    )

## 5. Short/long profiler evidence

Open the exported CPU/NPU timelines under `profiles/` in the lab's compatible
trace viewer. `homework.performance.prefill` and `homework.performance.decode` annotate the forward
scopes. The helper groups device-event names into projections/MLP, full
attention, convolution, GDN, normalization/activation, copies/layout, and
unclassified work. Inspect unclassified kernels before drawing conclusions.

Summed kernel durations can overlap. Union busy time and idle intervals are
reported separately; neither alone proves a bandwidth or host-dispatch
bottleneck. Missing NPU events are an unavailable profiling result, not evidence
that a category costs zero.

In [ ]:
profiles = json.loads((RUN / "profile_summary.json").read_text())
for profile in profiles:
    print(json.dumps(profile, indent=2))
if not profiles:
    print("No analyzable exported timeline: inspect profiles/ and worker.log")

## 6. Completion status

Keep `manifest.json`, exact `launch_config.json`, `fixtures.json`, raw requests,
forward samples, allocator/device snapshots and profiles with the report. A
complete default performance baseline needs all four cases, valid actual token
counts, at least 10 measured repetitions per case/pass, and short/long device
profile evidence. Performance results remain provisional while linked
correctness failures are outstanding.

In [ ]:
summary = json.loads((RUN / "summary.json").read_text())
print("Complete default timing/profile evidence:", summary["complete_default_evidence"])
print("All four cases measured:", summary["all_cases_measured"])
print("Ten-repetition baseline:", summary["ten_repetition_baseline"])
print(
    "Short/long NPU profiles observed:",
    summary["short_and_long_device_profiles_observed"],
)
print("Failed requests:", summary["failed_requests"])
print("Linked correctness:", summary["correctness"])
print("Artifacts:", RUN)